# Exploratory Data Analysis & RFM Segmentation

In this notebook, we will:
1. Load and explore the cleaned e-commerce dataset.
2. Perform Exploratory Data Analysis (EDA) to understand sales trends.
3. Calculate RFM (Recency, Frequency, Monetary) metrics for each customer.
4. Segment customers based on their RFM scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Sales over time (Monthly)
monthly_sales = df.resample('M', on='InvoiceDate')['TotalPrice'].sum()

plt.figure(figsize=(12, 5))
monthly_sales.plot(marker='o')
plt.title('Total Revenue per Month')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.show()

In [ ]:
# Sales by Category
cat_sales = df.groupby('Category')['TotalPrice'].sum().sort_values()

plt.figure(figsize=(8, 5))
cat_sales.plot(kind='barh', color='skyblue')
plt.title('Total Revenue by Category')
plt.xlabel('Revenue')
plt.show()

## 3. RFM Analysis

In [ ]:
# Snapshot date for Recency calculation
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate RFM
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique', # Frequency
    'TotalPrice': 'sum' # Monetary
})

rfm.rename(columns={'InvoiceDate': 'Recency',
                    'InvoiceNo': 'Frequency',
                    'TotalPrice': 'Monetary'}, inplace=True)

rfm.head()

## 4. RFM Scoring & Segmentation

In [ ]:
# Assign scores from 1 to 4
rfm['R_score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)

def segment_customer(df):
    if df['RFM_Score'] >= 10:
        return 'Champions'
    elif (df['RFM_Score'] >= 8) and (df['RFM_Score'] < 10):
        return 'Loyal Customers'
    elif (df['RFM_Score'] >= 6) and (df['RFM_Score'] < 8):
        return 'Potential Loyalist'
    elif (df['RFM_Score'] >= 4) and (df['RFM_Score'] < 6):
        return 'At Risk'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm.head()

In [ ]:
# Visualize Segments
segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='viridis')
plt.title('Customer Count by Segment')
plt.ylabel('Number of Customers')
plt.show()